# 차량통행속도 데이터 병합

TOPIS 도로·일자별 통행속도 월별 xlsx(`raw/speed/<연도>/*.xlsx`)를 연도별 parquet 하나로 합친다.

In [ ]:
from pathlib import Path

import pandas as pd

# notebooks/의 부모 = 프로젝트 루트. 실행 위치(cwd)와 무관하게 항상 이 파일 기준으로 찾는다.
ROOT = Path().resolve()
while not (ROOT / "CLAUDE.md").exists():
    ROOT = ROOT.parent

COLUMNS = [
    "일자", "요일", "도로명", "링크아이디", "시점명", "종점명", "방향",
    "거리", "차선수", "기능유형구분", "도심/외곽구분", "권역구분",
] + [f"{h:02d}시" for h in range(1, 25)]

In [ ]:
import re

HOUR_PATTERN = re.compile(r"^~?(\d{2})시$")


def _normalize_hour_columns(columns):
    """월별 파일마다 다른 '~01시'/'01시' 표기를 '01시'로 통일한다."""
    normalized = []
    for c in columns:
        m = HOUR_PATTERN.match(c)
        normalized.append(f"{m.group(1)}시" if m else c)
    return normalized


def merge_year(year: str) -> pd.DataFrame:
    """raw/speed/<year>의 월별 xlsx를 하나로 합쳐 검증 후 반환한다."""
    src_dir = ROOT / "raw" / "speed" / year
    files = sorted(src_dir.glob("*.xlsx"))
    if not files:
        raise FileNotFoundError(f"{src_dir}에 xlsx 파일이 없습니다.")

    frames = []
    total_rows = 0
    for f in files:
        df = pd.read_excel(f, engine="openpyxl")
        df.columns = _normalize_hour_columns(list(df.columns))
        if list(df.columns) != COLUMNS:
            print(f"[경고] 컬럼 불일치: {f.name}")
            print("  기대:", COLUMNS)
            print("  실제:", list(df.columns))
        n = len(df)
        total_rows += n
        print(f"{f.name}: {n:,}행")
        frames.append(df)

    merged = pd.concat(frames, ignore_index=True)
    assert len(merged) == total_rows, "합친 행 수가 원본 합과 다릅니다"
    assert list(merged.columns) == COLUMNS, "컬럼 정규화 후에도 기대 컬럼과 다릅니다"

    print(f"\n합계: {total_rows:,}행")
    print("일자 범위:", merged["일자"].min(), "~", merged["일자"].max())
    print("고유 일자 수:", merged["일자"].nunique())
    print("권역구분 고유값 수:", merged["권역구분"].nunique())
    return merged

In [ ]:
# 2019~2023은 헤더가 '01시'~'24시'로 정상이라 이미 만든 parquet을 다시 만들 필요 없음.
# 2024·2025만 '~01시' 표기 섞여 있어서 재생성 대상.
YEARS = ["2024", "2025"]
print("대상 연도:", YEARS)

for year in YEARS:
    merged = merge_year(year)
    out_path = ROOT / "work" / "speed" / f"speed_{year}.parquet"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    merged.to_parquet(out_path, index=False)
    print("저장:", out_path, "\n")